# Pipeline demo

Thin demo: reads the canonical Parquet output of `ffep ingest` and the latest per-run validation report. No mutation logic and no file writes live here -- run `ffep ingest` / `ffep run` to (re)produce the data this notebook reads. See `docs/pipeline.md` for the full CLI reference.

In [ ]:
import polars as pl
from pathlib import Path

PROCESSED = Path('../data/processed')
plays = pl.read_parquet(PROCESSED / 'plays.parquet')
games = pl.read_parquet(PROCESSED / 'games.parquet')
print(f'plays: {plays.height} rows, {plays.width} columns')
print(f'games: {games.height} rows, {games.width} columns')

## Row counts by source, competition and season

In [ ]:
plays.group_by(['source', 'competition', 'season']).len().sort(
    ['source', 'competition', 'season']
)

## Latest validation report -- quarantine section

Quarantined games are excluded from `plays.parquet`; the run continues with the remaining games. See `docs/pipeline.md` for the six per-game checks and the warn-only policy for the legacy source.

In [ ]:
report_path = PROCESSED / 'validation-report-latest.md'
report = report_path.read_text()
start = report.index('## Quarantined games')
end = report.index('## Summary')
print(report[start:end].strip())